# Agentic ServiceNow Search: IBM Orchestrate via MCP + Elastic

## Section 1 — Provisioning

In [ ]:
%%bash
echo "--- Initializing ---"
terraform -chdir=terraform init -upgrade -input=false > /dev/null && echo "Done."

echo "--- Applying Changes ---"
terraform -chdir=terraform apply -auto-approve > /dev/null && echo "Done."

echo "--- Exporting Environment Variables ---"
cat > .env << ENVEOF
ELASTIC_CLOUD_API_KEY=$(terraform -chdir=terraform output -raw elastic_cloud_api_key)
ELASTIC_CLOUD_ID=$(terraform -chdir=terraform output -raw cloud_id)
ELASTIC_ENDPOINT=$(terraform -chdir=terraform output -raw elasticsearch_endpoint)
KIBANA_URL=$(terraform -chdir=terraform output -raw kibana_endpoint)
ELASTIC_USERNAME=$(terraform -chdir=terraform output -raw elasticsearch_username)
ELASTIC_PASSWORD=$(terraform -chdir=terraform output -raw elasticsearch_password)
CONNECTOR_ID=$(terraform -chdir=terraform output -raw connector_id)
INDEX_NAME=$(terraform -chdir=terraform output -raw index_name)
SERVICENOW_URL=$(terraform -chdir=terraform output -raw servicenow_url)
SERVICENOW_USERNAME=$(terraform -chdir=terraform output -raw servicenow_username)
SERVICENOW_PASSWORD=$(terraform -chdir=terraform output -raw servicenow_password)
ORCHESTRATE_URL=$(terraform -chdir=terraform output -raw orchestrate_url)
ORCHESTRATE_API_KEY=$(terraform -chdir=terraform output -raw orchestrate_api_key)
ORCHESTRATE_AGENT_NAME=servicenow_search
MCP_SERVER_URL=$(terraform -chdir=terraform output -raw kibana_endpoint)/api/agent_builder/mcp
ENVEOF
echo "Done."


## Section 2 - Setup & Preflight

In [ ]:
from dotenv import load_dotenv
from orchestrate_relay.preflight import run_preflight

load_dotenv(override=True)
run_preflight()


## Section 3 — Sync

- Triggers a full sync via the connector API; the EC2 connector pulls records from ServiceNow and bulk-indexes them into Elasticsearch, with `semantic_text` fields vectorised automatically at index time

In [ ]:
%%bash
source .env

echo '--- Triggering full sync ---'
JOB_RESP=$(curl -sf -X POST "$ELASTIC_ENDPOINT/_connector/_sync_job" \
  -u "$ELASTIC_USERNAME:$ELASTIC_PASSWORD" \
  -H 'Content-Type: application/json' \
  -d "{\"id\": \"$CONNECTOR_ID\", \"job_type\": \"full\"}")
JOB_ID=$(echo "$JOB_RESP" | python3 -c "import sys,json; print(json.load(sys.stdin)['id'])")
echo "Sync job started: $JOB_ID"

echo '--- Polling for completion ---'
START=$(date +%s)
while true; do
    STATUS=$(curl -sf "$ELASTIC_ENDPOINT/_connector/_sync_job/$JOB_ID" \
      -u "$ELASTIC_USERNAME:$ELASTIC_PASSWORD" \
      | python3 -c "import sys,json; print(json.load(sys.stdin)['status'])")
    ELAPSED=$(( $(date +%s) - START ))
    printf '  %ds — %s\n' $ELAPSED $STATUS
    [ "$STATUS" = 'completed' ] || [ "$STATUS" = 'error' ] && break
    sleep 20
done

echo '--- Index doc count ---'
COUNT=$(curl -sf "$ELASTIC_ENDPOINT/$INDEX_NAME/_count" \
  -u "$ELASTIC_USERNAME:$ELASTIC_PASSWORD" \
  | python3 -c "import sys,json; print(json.load(sys.stdin)['count'])")
echo "$COUNT documents indexed in $INDEX_NAME"


## Section 4 - Semantic Query: Direct to Elasticsearch

In [ ]:
import os, time
from dotenv import load_dotenv
from elasticsearch import Elasticsearch
from IPython.display import Markdown, display

load_dotenv(override=True)

es    = Elasticsearch(
    os.environ['ELASTIC_ENDPOINT'],
    basic_auth=(os.environ['ELASTIC_USERNAME'], os.environ['ELASTIC_PASSWORD']),
)
IDX   = os.environ['INDEX_NAME']
QUERY = 'people cannot log in to company systems'

ESQL = f"""
FROM {IDX} METADATA _score
| WHERE (MATCH(short_description, ?query) OR MATCH(description, ?query))
  AND sys_class_name == "incident"
| KEEP number, sys_class_name, short_description, state, _score
| SORT _score DESC
| LIMIT 5
"""

t0   = time.time()
resp = es.esql.query(query=' '.join(ESQL.split()), params=[{'query': QUERY}])
elapsed_ms = (time.time() - t0) * 1000

cols = [c['name'] for c in resp['columns']]
rows = resp['values']

print(f'Query : "{QUERY}"')
print(f'Hits  : {len(rows)}')
print(f'Took  : {elapsed_ms:.0f} ms')

header = '| ' + ' | '.join(cols) + ' |'
sep    = '| ' + ' | '.join('---' for _ in cols) + ' |'
body   = '\n'.join(
    '| ' + ' | '.join(f'{v:.4f}' if isinstance(v, float) else str(v) for v in row) + ' |'
    for row in rows
)
md = f'\n{header}\n{sep}\n{body}'
print(md)

## Section 5 — Create & Verify ES|QL Search Tool



In [ ]:
# Section 5 — Create the ES|QL search tool in Agent Builder, then verify it
import json, os, time, requests
from pathlib import Path
from dotenv import load_dotenv
from IPython.display import Markdown, display

load_dotenv(override=True)

KIBANA   = os.environ['KIBANA_URL']
AUTH     = (os.environ['ELASTIC_USERNAME'], os.environ['ELASTIC_PASSWORD'])
HDRS     = {'kbn-xsrf': 'true', 'Content-Type': 'application/json'}
ENDPOINT = '/api/agent_builder/tools'
QUERY    = 'people cannot log in to company systems'

tool = json.loads((Path('orchestrate/tools') / 'search_servicenow.json').read_text())

# Idempotent: delete-then-create
requests.delete(f'{KIBANA}{ENDPOINT}/{tool["id"]}', auth=AUTH, headers=HDRS)
resp = requests.post(f'{KIBANA}{ENDPOINT}', auth=AUTH, headers=HDRS, json=tool)
resp.raise_for_status()
print(f'Tool "{tool["id"]}" registered.')

# Verify: execute the tool directly before involving any MCP machinery
t0 = time.time()
ex = requests.post(
    f'{KIBANA}{ENDPOINT}/_execute',
    auth=AUTH, headers=HDRS,
    json={'tool_id': tool['id'], 'tool_params': {'query': QUERY}},
    timeout=30,
)
ex.raise_for_status()
elapsed_ms = (time.time() - t0) * 1000

body = ex.json()
esql = next(r for r in body['results'] if r['type'] == 'esql_results')
cols = [c['name'] for c in esql['data']['columns']]
rows = esql['data']['values']

print(f'\nTool test  query : "{QUERY}"')
print(f'           hits  : {len(rows)}   took : {elapsed_ms:.0f} ms')

header = '| ' + ' | '.join(cols) + ' |'
sep    = '| ' + ' | '.join('---' for _ in cols) + ' |'
body_md = '\n'.join(
    '| ' + ' | '.join(f'{v:.4f}' if isinstance(v, float) else str(v) for v in row) + ' |'
    for row in rows
)
md = f'\n{header}\n{sep}\n{body_md}'
print(md)

## Section 6 — Wire Orchestrate

In [ ]:
%%bash
# Section 6 — Register Orchestrate connection, toolkit, and agent
source .env

echo "--- Activating Orchestrate environment ---"
echo Y | orchestrate env add --name orchestrate-relay --url "$ORCHESTRATE_URL"
orchestrate env activate orchestrate-relay --api-key "$ORCHESTRATE_API_KEY"
echo ""

echo "--- Registering MCP connection (API key) ---"
orchestrate connections add -a elastic_mcp >/dev/null 2>&1 || true
orchestrate connections configure -a elastic_mcp \
  --env draft -t team -k basic -u "$MCP_SERVER_URL"
orchestrate connections set-credentials -a elastic_mcp --env draft \
  --username "$ELASTIC_USERNAME" \
  --password "$ELASTIC_PASSWORD"
echo ""

echo "--- Registering MCP toolkit (idempotent) ---"
orchestrate toolkits remove -n elastic-agent-builder >/dev/null 2>&1 || true
orchestrate toolkits add -k mcp \
  -n elastic-agent-builder \
  --description 'ServiceNow semantic search via Elastic Agent Builder' \
  -u "$MCP_SERVER_URL" --transport streamable_http \
  -t '*' -a elastic_mcp
echo ""

echo "--- Importing agent ---"
orchestrate agents import -f orchestrate/agent.yaml

## Section 7 — Ask via Orchestrate

In [ ]:
%%bash
# Section 7 — Ask the question via Orchestrate
source .env
orchestrate env activate orchestrate-relay --api-key "$ORCHESTRATE_API_KEY" > /dev/null

QUERY='people cannot log in to company systems'
echo "Query: $QUERY"
echo ''

# pipe "exit" so the interactive loop exits cleanly after printing the response
echo "exit" | orchestrate chat ask \
  --agent-name "$ORCHESTRATE_AGENT_NAME" \
  --include-reasoning \
  "$QUERY"

## Section 8 - Teardown

In [ ]:
%%bash
echo "--- Destroying Infrastructure ---"
terraform -chdir=terraform destroy -auto-approve > /dev/null && echo "Done."

echo "--- Cleaning Up ---"
rm -f .env && echo "Done."
